In [13]:
# ============================================================
# UCI-HAR Fine-tuning with ImageNet Pretrained InceptionV3
#
# 1. Pretrained InceptionV3 + Linear Probing
# 2. Unfreeze All Layers + Full Fine-tuning
# 3. Final Test Evaluation
#
# Google Colab-ready
# ============================================================

import os
import glob
from pathlib import Path

import numpy as np
import tensorflow as tf

from google.colab import drive

from tensorflow.keras import backend as K
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# ============================================================
# 1. Reset and Mount Drive
# ============================================================

K.clear_session()

drive.mount("/content/drive")


# ============================================================
# 2. Basic Settings
# ============================================================

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SIGNAL_NAMES = [
    "body_acc_x_",
    "body_acc_y_",
    "body_acc_z_",
    "body_gyro_x_",
    "body_gyro_y_",
    "body_gyro_z_",
    "total_acc_x_",
    "total_acc_y_",
    "total_acc_z_"
]

LABELS = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

NUM_CLASSES = 6
BATCH_SIZE = 64

LINEAR_PROBE_EPOCHS = 30
FINETUNE_EPOCHS = 40

IMG_SIZE = 128

LR_LINEAR_PROBE = 1e-3
LR_FINETUNE = 1e-5


# ============================================================
# 3. Robust UCI-HAR Path Detection
# ============================================================
# This supports:
# - train/Inertial Signals/body_acc_x_train.txt
# - train/Inertial_Signals/body_acc_x_train.txt
# ============================================================

def detect_uci_har_base_dir():
    target_file = "body_acc_x_train.txt"

    search_roots = [
        "/content/drive/MyDrive",
        "/content/drive/Shareddrives",
        "/content/drive"
    ]

    matched_files = []

    for root in search_roots:
        if os.path.exists(root):
            matched_files.extend(
                glob.glob(
                    os.path.join(root, "**", target_file),
                    recursive=True
                )
            )

    matched_files = sorted(list(set(matched_files)))

    if len(matched_files) == 0:
        raise FileNotFoundError(
            "body_acc_x_train.txt 파일을 찾지 못했습니다. "
            "Google Drive에 UCI-HAR 데이터셋이 압축 해제되어 있는지 확인하십시오."
        )

    body_acc_path = Path(matched_files[0])

    # Example:
    # /content/drive/MyDrive/.../UCI-HAR/UCI-HAR/train/Inertial Signals/body_acc_x_train.txt
    #
    # parents[0] = Inertial Signals
    # parents[1] = train
    # parents[2] = UCI-HAR root
    base_dir = body_acc_path.parents[2]

    print("Found UCI-HAR signal file:")
    print(body_acc_path)

    print("\nDetected BASE_DIR:")
    print(base_dir)

    return str(base_dir)


BASE_DIR = detect_uci_har_base_dir()


def get_inertial_signal_dir(split):
    candidates = [
        os.path.join(BASE_DIR, split, "Inertial Signals"),
        os.path.join(BASE_DIR, split, "Inertial_Signals")
    ]

    for signal_dir in candidates:
        if os.path.exists(signal_dir):
            return signal_dir

    raise FileNotFoundError(
        f"{split} 폴더 안에서 'Inertial Signals' 또는 'Inertial_Signals' 폴더를 찾지 못했습니다."
    )


# ============================================================
# 4. Load UCI-HAR Data
# ============================================================

def load_signals(split):
    signal_dir = get_inertial_signal_dir(split)
    signals = []

    print(f"\nLoading {split} signals from:")
    print(signal_dir)

    for name in SIGNAL_NAMES:
        file_path = os.path.join(signal_dir, f"{name}{split}.txt")

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Missing signal file: {file_path}")

        data = np.loadtxt(file_path)
        signals.append(data)

    signals = np.stack(signals, axis=-1).astype(np.float32)

    return signals


def load_labels(split):
    label_path = os.path.join(BASE_DIR, split, f"y_{split}.txt")

    if not os.path.exists(label_path):
        raise FileNotFoundError(f"Missing label file: {label_path}")

    labels = np.loadtxt(label_path).astype(np.int64)

    # UCI-HAR labels are 1~6, convert to 0~5
    labels = labels - 1

    return labels


X_train = load_signals("train")
y_train = load_labels("train")

X_test = load_signals("test")
y_test = load_labels("test")

print("\nOriginal train:", X_train.shape, y_train.shape)
print("Original test :", X_test.shape, y_test.shape)


# ============================================================
# 5. Train-only Standardization
# ============================================================

N_train, T, C = X_train.shape
N_test = X_test.shape[0]

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, C)
X_test_2d = X_test.reshape(-1, C)

X_train = scaler.fit_transform(X_train_2d).reshape(N_train, T, C).astype(np.float32)
X_test = scaler.transform(X_test_2d).reshape(N_test, T, C).astype(np.float32)

print("\nAfter standardization:", X_train.shape, X_test.shape)


# ============================================================
# 6. Convert Time-Series Window to InceptionV3 Input Image
# ============================================================
# Original UCI-HAR window : (128, 9)
# InceptionV3 input       : (128, 128, 3)
#
# Step:
# 1) expand channel: (128, 9, 1)
# 2) resize to      : (128, 128, 1)
# 3) repeat channel : (128, 128, 3)
# 4) convert to image-like range and apply InceptionV3 preprocess_input
# ============================================================

def timeseries_to_inception_image(x, y):
    x = tf.expand_dims(x, axis=-1)                  # (128, 9, 1)
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))    # (128, 128, 1)
    x = tf.repeat(x, repeats=3, axis=-1)            # (128, 128, 3)

    x = tf.clip_by_value(x, -3.0, 3.0)
    x = (x + 3.0) / 6.0
    x = x * 255.0

    x = preprocess_input(x)

    return x, y


def build_dataset(X, y, training=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if training:
        ds = ds.shuffle(
            buffer_size=len(X),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        timeseries_to_inception_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = build_dataset(X_train, y_train, training=True)
test_ds = build_dataset(X_test, y_test, training=False)


# ============================================================
# 7. Build ImageNet Pretrained InceptionV3 Classifier
# ============================================================

def build_pretrained_inceptionv3_model(num_classes=6):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    base_model = InceptionV3(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    # Stage 1: freeze backbone for linear probing
    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="Pretrained_InceptionV3_UCIHAR"
    )

    return model, base_model


model, base_model = build_pretrained_inceptionv3_model(num_classes=NUM_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_LINEAR_PROBE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# ============================================================
# 8. Stage 1: Linear Probing
# ============================================================
# InceptionV3 backbone is frozen.
# Only the classification head is trained.
# ============================================================

print("\n================ Stage 1: Linear Probing ================\n")

callbacks_stage1 = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    )
]

history_linear_probe = model.fit(
    train_ds,
    epochs=LINEAR_PROBE_EPOCHS,
    callbacks=callbacks_stage1,
    verbose=1
)


# ============================================================
# 9. Stage 2: Full Fine-tuning
# ============================================================
# All InceptionV3 layers are unfrozen.
# The whole model is fine-tuned using a small learning rate.
# ============================================================

base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n================ Stage 2: Full Fine-tuning ================\n")

callbacks_stage2 = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

history_finetune = model.fit(
    train_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=callbacks_stage2,
    verbose=1
)


# ============================================================
# 10. Final Test Evaluation
# ============================================================

print("\n================ Final Test Evaluation ================\n")

y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy : {acc:.4f}")
print(f"Macro-F1 : {macro_f1:.4f}")

print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=LABELS,
        digits=4
    )
)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found UCI-HAR signal file:
/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/UCI HAR Dataset/train/Inertial Signals/body_acc_x_train.txt

Detected BASE_DIR:
/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/UCI HAR Dataset

Loading train signals from:
/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/UCI HAR Dataset/train/Inertial Signals

Loading test signals from:
/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/UCI HAR Dataset/test/Inertial Signals

Original train: (7352, 128, 9) (7352,)
Original test : (2947, 128, 9) (2947,)

After standardization: (7352, 128, 9) (2947, 128, 9)
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "Pretrained_InceptionV3_UCIHAR"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 63, 63,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 63, 63,    │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 63, 63,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 61, 61,    │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 61, 61,    │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 61, 61,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 61, 61,    │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 61, 61,    │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 61, 61,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 30, 30,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 30, 30,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 30, 30,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 30, 30,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 28, 28,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 28, 28,    │          0 │ batch_normalizat

 Total params: 22,360,998 (85.30 MB)

 Trainable params: 558,214 (2.13 MB)

 Non-trainable params: 21,802,784 (83.17 MB)


================ Stage 1: Linear Probing ================

Epoch 1/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 33s 121ms/step - accuracy: 0.6634 - loss: 0.9048 - learning_rate: 0.0010
Epoch 2/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.7991 - loss: 0.5323 - learning_rate: 0.0010
Epoch 3/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.8369 - loss: 0.4201 - learning_rate: 0.0010
Epoch 4/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.8557 - loss: 0.3668 - learning_rate: 0.0010
Epoch 5/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.8652 - loss: 0.3491 - learning_rate: 0.0010
Epoch 6/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.8754 - loss: 0.3246 - learning_rate: 0.0010
Epoch 7/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.8781 - loss: 0.3110 - learning_rate: 0.0010
Epoch 8/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.8920 - loss: 0.2842 - learning_rate: 0.0010
Epoch 9/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/st